# Caching Strategies [Security - Module 03]

> **MLCourse - Agentic AI - Production Security**

Caching reduces cost, latency, and load on LLM providers. But caching has
a security side: what you cache, where you cache it, and who can read
cached data all matter. This module covers the main caching layers for
LLM agents -- exact-match, semantic, and response caching -- with emphasis
on cache keys, invalidation, and the security pitfalls of caching sensitive
data. Everything runs without an API key.

### What you will learn

1. Why cache LLM calls (cost, latency, load).
2. Exact-match (key-value) caching and cache keys.
3. Semantic caching using embeddings and similarity.
4. Time-to-live (TTL) and invalidation strategies.
5. What is safe to cache vs. what must never be cached.
6. Cache poisoning and cache-busting on sensitive operations.
7. Building a small, safe TTL cache.

### Key takeaways

- A cache key must capture the inputs that change the answer.
- Never cache secrets, PII, or per-user private data in a shared cache.
- TTL bounds staleness; invalidate on changes.
- Cache poison risk grows with untrusted input.
- Semantic caching reuses embeddings + similarity.

### Setup: imports, environment


In [ ]:
import os
import time
import json
import hashlib
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives inside the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)
print("Module 03: Caching Strategies")
print(f"Track root: {TRACK}")
print("Note: caching examples are deterministic (no API key needed).")


### Verify embeddings availability for semantic caching


In [ ]:
EMB_OK = False
try:
    from sentence_transformers import SentenceTransformer
    _emb = SentenceTransformer("all-MiniLM-L6-v2")
    EMB_OK = True
    print("SentenceTransformers: AVAILABLE (all-MiniLM-L6-v2)")
except Exception as e:
    print("SentenceTransformers: OFFLINE --", e)
    print("Semantic caching will use a hashing fallback for the demo.")


### 1. Why Cache LLM Calls

Every LLM call costs money, takes time, and consumes a quota. Many agent
workloads repeat questions or ask semantically similar questions:

- A support bot gets the same FAQ asked repeatedly.
- A RAG agent re-asks the same questions with different wording.
- Multi-user systems see heavy overlap.

Caching the generated answer (or the retrieval step) for repeated queries
turns those repeated calls into near-instant, free lookups.

### Cost / latency motivation


In [ ]:
print("=== Why cache ===\n")
print("  1 token ~ 0.75 word; a typical answer ~ 200-500 tokens")
print("  1000 repeated queries * 300 tokens = 300k tokens wasted")
print("  Cache hit latency: ~1ms (embedding lookup)")
print("  Cache miss latency: 1-5s (full generation)")
print("\nFinancial + latency win, plus provider quotas.")


### 2. Exact-Match Caching and Cache Keys

The simplest cache maps an exact key to a value. The **key must capture
everything that changes the answer**: the model, the system prompt, the
temperature, the full user message, and any relevant context. If you
forget a parameter in the key, users get the wrong (possibly unsafe)
cached answer.

### Exact-match cache with a proper composite key


In [ ]:
@dataclass
class ExactCache:
    data: Dict[str, Any] = field(default_factory=dict)
    def _key(self, model, system, message, temperature, context=""):
        raw = json.dumps([model, system, message, temperature, context],
                         sort_keys=True)
        return hashlib.sha256(raw.encode()).hexdigest()[:16]
    def get(self, *args):
        k = self._key(*args)
        return self.data.get(k)
    def put(self, value, *args):
        k = self._key(*args)
        self.data[k] = value

cache = ExactCache()
cache.put("Refund policy answer", "gpt-4o", "support", "What is the refund policy?", 0.0, "")
print("Stored, key:", list(cache.data.keys())[0])
# Same key -> hits
hit = cache.get("gpt-4o", "support", "What is the refund policy?", 0.0, "")
print("Exact match hit:", hit)
# Different temperature -> miss (correct!)
miss = cache.get("gpt-4o", "support", "What is the refund policy?", 0.7, "")
print("Different temperature -> miss:", miss)


### 3. Semantic Caching

Exact-match caching misses when users rephrase. Semantic caching embeds
the query and stores it alongside the answer; a new query hits the cache
if its embedding is similar (above a threshold) to a stored query.

This reuses the concept from the RAG caching module but frames it as a
general agent-layer cache and highlights the security angle.

### Semantic cache using embeddings (with deterministic fallback)


In [ ]:
def _embed(text: str):
    if EMB_OK:
        return _emb.encode(text).astype(np.float32)
    # No-model fallback: hash-based pseudo-embedding so the module runs.
    vec = np.zeros(8, dtype=np.float32)
    for ch in text:
        vec[hash(ch) % 8] += 1.0
    return vec

def _cos(a, b):
    a, b = a / max(np.linalg.norm(a), 1e-9), b / max(np.linalg.norm(b), 1e-9)
    return float(np.dot(a, b))

@dataclass
class SemanticCache:
    threshold: float = 0.85
    entries: List = field(default_factory=list)  # [ ({vec,answer,ts} ]
    def lookup(self, text):
        v = _embed(text)
        best, best_sim = None, -1.0
        for e in self.entries:
            s = _cos(v, e["vec"])
            if s > best_sim:
                best_sim, best = s, e
        if best is not None and best_sim >= self.threshold:
            return best["answer"], best_sim
        return None, best_sim
    def put(self, text, answer):
        self.entries.append({"vec": _embed(text), "answer": answer,
                             "ts": time.time()})

scache = SemanticCache(threshold=0.85)
scache.put("What is the refund policy?", "A: 30 day returns")
hit, sim = scache.lookup("What is the return policy?")
print(f"Semantic lookup similarity={sim:.3f}")
print(f"Hit: {hit if hit else 'MISS'}")


### 4. TTL and Invalidation

Cached answers go stale. A policy may change, data may update, or the
model/prompt may change. Two mechanisms keep caches fresh:

- **TTL (Time To Live)**: entries expire after a duration.
- **Invalidation**: explicitly remove entries when the source changes.

For dynamic or sensitive data, prefer active invalidation over long TTL.

### TTL cache


In [ ]:
@dataclass
class TTLCache:
    ttl: float = 60.0
    data: Dict = field(default_factory=dict)
    def get(self, key):
        item = self.data.get(key)
        if item and (time.time() - item["ts"]) < self.ttl:
            return item["value"]
        if key in self.data:
            del self.data[key]
        return None
    def put(self, key, value):
        self.data[key] = {"value": value, "ts": time.time()}

tc = TTLCache(ttl=2)
tc.put("k1", "v1")
print("Immediate:", tc.get("k1"))
time.sleep(3)
print("After TTL :", tc.get("k1"))


### 5. What Must Never Be Cached

This is the security-critical part. A shared cache (in-memory, Redis,
edge CDN) is visible to the whole process or service. Putting private
data in it leaks that data to others and to logs.

A cache key containing your API key, auth tokens, or the user's PII
effectively stores that secret. The cached **value** containing PII,
a refund decision, medical info, or financial data is equally dangerous.

### Cache safety checklist


In [ ]:
print("=== Cache safety checklist ===\n")
unsafe = [
    ("API keys / tokens", "key or value", "store in secret manager, not cache"),
    ("PII (name, email, SSN)", "value", "do not cache user-specific data"),
    ("Per-user private answers", "value", "use per-user isolated stores"),
    ("Temporary one-time codes", "value", "never persist"),
    ("Prompts containing secrets", "key/value", "strip secrets before hashing"),
]
for what, where, fix in unsafe:
    print(f"  [AVOID] {what:32s} ({where:12s}) -> {fix}")

print("\nRule: if an entry is not safe to log, it is not safe to cache.")


### 6. Cache Poisoning and Cache-Busting

**Cache poisoning**: an attacker submits an input that gets cached with a
dangerous or incorrect answer, which is then served to all subsequent
users. This amplifies harm and bypasses per-request guardrails.

**Mitigations**:
- Only cache **validated** responses (post guardrail).
- Never cache responses that were refused or flagged.
- Include the policy version in the key.
- **Cache-bust** (invalidate) after any policy or data change.

**Cache-busting** is a defense: after a policy update you must purge
affected entries so stale (or poisoned) answers are not served.

### Demonstrate poisoning + cache-busting


In [ ]:
class GuardedTTLCache(TTLCache):
    def __init__(self, ttl=300):
        super().__init__(ttl=ttl)
        self.blocked = set()
    def put_if_safe(self, key, value, safe: bool):
        if not safe:
            self.blocked.add(key)   # never cache unsafe responses
            return False
        self.put(key, value)
        return True
    def invalidate(self, key):
        self.data.pop(key, None)
        self.blocked.discard(key)

safe_cache = GuardedTTLCache()

# An unsafe (poisoned candidate) response is NOT cached
stored = safe_cache.put_if_safe("policy_q", "ALL REFUNDS APPROVED", safe=False)
print("Unsafe response cached?", stored, "| blocked keys:", list(safe_cache.blocked))

# A safe response is cached
safe_cache.put_if_safe("policy_q", "Refunds need review", safe=True)
print("Safe cached:", safe_cache.get("policy_q"))

# Policy changed -> invalidate (cache-bust)
safe_cache.invalidate("policy_q")
print("After invalidate:", safe_cache.get("policy_q"))


### 7. Composing Caching and Guardrails

The correct order is: **guard, then cache**. The response must pass all
guardrails before it is eligible for caching. This prevents poison from
propagating and keeps cached values safe to serve to any user.

### Integrated guarded cache


In [ ]:
class GuardedCachePipeline:
    def __init__(self, ttl=300):
        self.cache = GuardedTTLCache(ttl=ttl)
        # reuse a simple safety check
        self.unsafe = {"refund", "pay", "password", "key", "secret"}
    def answer(self, query):
        hit = self.cache.get(query)
        if hit is not None:
            return hit, "cache-hit"
        # simulate generation + safety verdict
        is_safe = not any(w in query.lower() for w in self.unsafe)
        answer = f"answer-for:{query}"
        self.cache.put_if_safe(query, answer, safe=is_safe)
        return ("cached-by-policy" if is_safe else "not-cached"), "generated"

gcp = GuardedCachePipeline()
print("Q1 (safe)  :", gcp.answer("What is the weather?"))
print("Q1 again   :", gcp.answer("What is the weather?"))  # hit
print("Q2 (unsafe):", gcp.answer("What is my password?"))
print("Q2 again   :", gcp.answer("What is my password?"))  # never cached


### 8. Cache Layers Summary

| Cache | Key | Pros | Security concern |
|-------|-----|------|------------------|
| Exact | full param hash | simple, exact | misses rephrases |
| Semantic | embedding | handles rephrasing | needs embedding model |
| TTL | key + expiry | bounds staleness | still needs invalidation |

Always: guard first, cache second; never cache secrets/PII; bust on change.

### Final summary


In [ ]:
print("=== Module 03 Summary ===\n")
summary = [
    "Exact-match: composite key capturing every answer-affecting input.",
    "Semantic: embed query, reuse answer when similar.",
    "TTL: expire stale entries automatically.",
    "Invalidation: purge on policy/data change (cache-bust).",
    "Never cache: secrets, PII, per-user private data, unsafe responses.",
    "Order: guard -> validate -> THEN cache.",
]
for s in summary:
    print("  -", s)
